# G11project — One-run Demo-Lite Training v2

Use **Runtime → Run all** with an A100/T4 GPU. This notebook mounts Drive, installs a current SUMO wheel, applies the Colab compatibility fix to the exact executed script, validates one complete SUMO scenario, runs lightweight Transformer-PPO training, compares it with three baselines, and exports presentation-ready results.

Scope: preliminary highway-only course-demo evidence. It is intentionally smaller than a paper-scale experiment.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import base64
import io
import importlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Select a GPU runtime, then use Runtime -> Run all"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

REPOSITORY_URL = "https://github.com/yangyu-rgb/G11project.git"
REPOSITORY = Path("/content/G11project").resolve()
DEMO_LITE_ROOT = Path("/content/drive/MyDrive/G11project-demo-lite-v2").resolve()
DEMO_LITE_WORK = Path("/content/g11-demo-lite-v2-work").resolve()
PIPELINE = REPOSITORY / "BackEnd/scripts/run_demo_lite_pipeline.py"
GENERATOR = REPOSITORY / "BackEnd/scripts/generate_highway_scenario.py"

if not (REPOSITORY / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    print("Reusing repository already present in this runtime:", REPOSITORY)
os.chdir(REPOSITORY)
DEMO_LITE_ROOT.mkdir(parents=True, exist_ok=True)
DEMO_LITE_WORK.mkdir(parents=True, exist_ok=True)

# These three demo-lite scripts are embedded so a fresh runtime does not depend
# on uncommitted local files or an updated GitHub branch.
BOOTSTRAP_B64 = "UEsDBBQAAAAIAEWr+1zslq6fKQkAAAwgAAApABwAQmFja0VuZC9zY3JpcHRzL3J1bl9kZW1vX2xpdGVfcGlwZWxpbmUucHlVVAkAA3JcZ2p0XGdqdXgLAAEE9QEAAAQUAAAArVlLc+O4Eb7rV2CZw5JblMaZzauUKBVnx8khqdmpnUkuLhUKJiEJGb4WAG1rXf7v6QZIEASphyvRwSbB7kbj6yeAKIp+aiuiD5yIKucNhz+VTonkqi3ZQ8HJQewPT+y4rKviSLK6lYovc17WhD83XIoSyEkjGl6Iiq+iKFosdrIuCaW7VreSU0pE2dRSE1ZVtWZa1JVaLPoxuW8YSOzfs7o59s//UXXVP6ujsmIbpg+FeOhlfoJX+0EfG1Ht+/Hb6ujm0LXMDovFX29/+Mfdxw/0px9//EI2hjMGJUUBKiYrWG9dPPI4WYE+sCR1/+vtQuyI0jL2ORMCiwCoUKMVKrNeEPj1bytRKS51fJNOOZMOGZVJ0Wi1emA6O1AtGUjrVJVtRc0wIb+CiX5ma3L3m5v3Y749r7hkmlO0Ai0EPKG1Cq16MWcozsjFybO6hPULQN5XyRud45fZavAFtRK1s7iuS5HRJ4kaoDlTsheallyznGmWkqJmOT2ysgjkLj5/uf373WcwUxwZgMC0UUqiQRF869YUAbCLrGBKkQ+w3n/CZJ9Yq3geg2drUOpOylom1lDgoJ/FvmIFYTChLOGhQWLCdppLAqtQQmn0JFYAzSMThYmCRtZ7mE9ZDw+n69zfTpHzHXg/6Kwpjc2I8RBe7FL31s0DeFFZ13pt3HH4/FTLr7MfvhsecWX0oc33HBAVVau5WpMdIKoHGlhD/USzpl2Th7ouAM+/sUJxS5CQ5Z/Jx7pXu1dyFegGTMEIWptVOaAmYy9yxlLcEoDfPV/FGc5Wfs2FjLuw3HyRLU8h9wAFrb+a11Mzv5lzBlLQfmZ0zOZQBmL3PCZRGsMRMwTQzKL8jkRDtBryFUZMNCMHw8IN4w8JVxhLKg5mA4whwjR/1jGvsjoHx95Erd4t/xAlyUgG5rqAVyiTHuMxIQcHIi8R0O25itbk5fXVfQ/wNEEgWCF+6ZYEooYQmXxEnhmnzFqJBqSQOmDlfgLxNGskfxR1q2hZ57zH2MIIdosjTGP4KRpYYMVjLvDMYOSbjWeVaD3CAdISAOGnmDgajEqMUR94UVd7BSUI0k0udjuOK/ErZ6CTmx4yXWnWGy4E1g+57+U1sa+WzpPQo+UEePCdYPGB6NgQikDSCJ1u7JtNQHURotF3/Bl8l4ivy7GnYEMHsNP8kZiUTSr+RD5I8cgN3NFIeDITOau2ydHTRoQvU52ct6x9+6dTQgUNi6EadUkAUyEgVTDZd0wEpcwJQHOufRvN0EgLIMba5KMhaI76AKlibdqQR/RByAeqAaXj5P5mOxVpuExfBEzm/4rSjpHSE/RZmzPqKqJjxGFMFe5LnJwQsG/aMRdm1Bw8KuO0YiWPbxL0xZNiberB5DCd4HU8dDolKfY4TkNm4ETqmTQwYX5NPefyhOJ7qwapucj0PTSEKTamW6/emiQ6jnLFNYhg0NnEfZI18e6YJIe+ugp8wXfZQdj9MBx4gfNcn9iOhZTWRX06HAmpBicNpjejk9ld9UDDrzsgTH7CgdQWGMAQYws3JdgAviZkV0uCBNiB2x4xMHxUQa2jRhyw4ss05ZgpZkSZGjhRpM+0nT6JKQrYiBZc8zyacfaph3pEr4OXADjGRXC39XPLoW7kVnPEQ152HN9vwNuSe3/xW1xNIJdsNhagyIZS8NkvCJZeqCAgPP8bTT1l7bZJFtm5svBvVrRdUdhFbfW1qp8qMhSDDogX8/81mpkBF+M2B6Y84Zyns0dPMe7ZLhf1Qad+NgOcgFIFJeiHf324Jb2bz9Wc+97Zt/fmYQtm870beM0KXucTVT+o5TE0A+5+sCOCbkjDZtO60i6iGPMON08Ef854o4NN0ljo9WqbXVPuaX1Sc4frteBMUkU35RB0Ye4xUPSph6q2LKH0xkZi2gE1CsGz+s76t/n6F3wVGbSfhzr3Kog/n4ndftJ1EMBnY3oneJGrKQDOx9dkpn3CXjLDEAarq7kOw6F2hqKC2r+HDQKneGwwS3NgZSNw330qsbnJug36BW0NPrD1ebqk9HnCHcR3K3n4LdSrPyoApboql56cBISKvZHp+0znn5Mq/JUf1521TZ2Ad1um4AEToLWql/VNePY2vdwigHFAHfAKd1aCHTyOqXfeCRLIGk57VkgW+R7dpQp3vjQ2jZWXTkN5ZpPaNbpWTIC6OTTkdqMdlgx5pGgrns98Njv2utVNqzfB4cGFGWf25ptTW/mJSgAbhRVSu3z6AKUwUG5UdTwj99kvmase4/w6lXAfWSBsPG4vFqDdXAUCnVF/RfoAgFLZif/WE//tNiidvQpBVthiV9N/G6eULcFGCUzhWPtUcFn1Oc1zYcsw7PbyNsNdXK8LcZInydjO7QXRkGg6870hkvCQeZVz3uBDPBNXnvAwppJA3H2E3XWBBax31AiLGJ7+vj2e3vUIvDNSjUuufhFNNOORHTJWjSC9DAu4DMsoOXiMM4V9CnpyXdLwGPtMfEXmmHVcl/DfFjjeEfal0MGha+LGqxknwseneEuseMqWTEvxjI24qPq5rwgPtGWH9DUeYIScvjW4zpMboIZ3c8NzwszeNYFh28zJ8h1gscDlmCsiyuS+NayxWU1/dbT6CPs01bCs26aYQYlHsT3Bbcf4yXyJc25vPEDNDaV5nVGaeJwrludurjhaLu1+KoXsVIsM6ktst1Ap+c5ubpKUdBv2jf1yVtqw2KU5s0r7zURXGPEii2/w1P+sGKyQ/5MAdLylLZDLrkBGHa+9RHCL+v53N5fwwQY9JSwzmELHXksObUnLz0Nh9mBL2IPN86bkwItmE5kjNeyoijpjBQFFzZ2P57mdfOcmpmE3nlNCoFt3EVV3OOncCC82QsdadOHuhmbvBP60Id/fDPHTSIELQkpiKUnZKjz9JXvJGd4r6QOrgIcMSOPJ+gbP65TOuZSTkH5vketulkDX8LJpCMpB2ZORNJC41ursTdLmLAAzF0wegxvrrpmmmFqPCQE0txh5WzYqdvfJ/f4rNbfTld68TwmvFN4rM5UJsTH3WckEvRszMNozuzrnhOP5y0ipfXchdGGj/P9X9/e/XcwKtjpfJc0tfIHX1tQcqFJqDkkoxTCgtLvAsLXn8xH8pLx7Fjq2QZIs/gtQSwMEFAAAAAgAkav7XApj96J4CgAAvBwAAC0AHABCYWNrRW5kL3NjcmlwdHMvZ2VuZXJhdGVfZGVtb19saXRlX3Jlc3VsdHMucHlVVAkAAwFdZ2oDXWdqdXgLAAEE9QEAAAQUAAAApVnpchu5Ef7Pp0DgHztjD8c6bGfNKqZKlrhZZy1LJcmpSimqETgDkrDnWgCUxNXqofIKebJ045iDh7y10Q97ZoBu9Pl1N0gp/TsvuWSak1pyxUvNtKjKoeQsWxEmtZixVCsyqyTRC04WYr64Z6thVeYrkvGiGuYCaPlDzaUogDymlA4GM1kVJElmS72UPEmIKOpKasLKsrIHqMHAf5PzmknF/Xuq7vzjV1WV/lmtlOVaM73IxdSzPIdXu6BXtSjn/vtRuWqOKJiu80oD1WDQPsdLxQN6NJ/TcHNjXK/wiTBF6lwT8oKU1a9sRCZv9g787nJZ1CvcUdZrGwYfjo5/mXw+SS7Ozq7I2AgZgDlEDsYIYzB0ld/xIIxBc7CZut6/GYgZUVoGXcoQeGoiSlQ+Rr1HAwJ//i0WpeJSB3vRJmXofKBkGrfOUbGoGl/oqhBpci/BfwlaOiJTln7jZZYg84jMhU4KrlnGNFvX73Ry9fPZySWoFlAmaEToVFYsS5nS+JIJpVmZcnxeyjkv0xUY+fjs09kF0jwaNZBwROiLg7fvDicfaGQ/tnxw7eT44N3BO7/WsMWln96+n+w1ZP4UXNl/d3T45ghWngZXRx8+TRKQ9uLjsZHWnTyb8VTzLLnjC5GCU9LqDpJgzj07bjaIO55kPIf/5CrBHPHL9fu3SQ7vcGJSKP81rYpiWYrUBHiCHBeQRX5VgwfyVcLvwA+eGdjk56OLqw0Bg+ckJPTILQ7dIukuXiDrMHJ81iUl9Pz9W+K+4OupyHOheFqVmWrJdqlC6HF3hXRXTrlSIIMir0ljPuLNh6whJgcZn5EkBxebkAtMUJv0CMnwbyQTqb6GWI4wfW9stGu5sg/4d8fyJQczIXGMbJRhESNcJZo/6AD0qjLAgTFd6tnwRwqZgIT8IeW1JsHZ5UTKCg4wHP5xefb5BHTPuPkaYjLDzvY8ycA25J94qtkRzGghlEKcAUAUJcgjsg4OQmYvcz0ijyjVEw2JSUJgaThCipuEBnobyIHRJzJ6h8+eun4EKZZKkykHSCXV9CtYuz3U8JEcoLe0BvN2LzgrA7UsCgY2XTN2RCDXpUhHiCXGGTOwrya/k89Vya1s3vqORTznOrBUEXl8Cs07xUP6MhhGVtUQbWDZCGWMgdwJz0FffPKSWlTSbJrz3QKvBU8r573QC7MaVzUvA3oP4Vny+1yUfEzheSNG0O/AmLOidYIRQYK2UJBi+xLYPeHaHrda3QeovV5UmA4ve9ATtjRYSe0uRHYHpO2x+GdtqlpLXzu+it5c26ebHsG6HL1Fk9EbX9w5wCrauvZyOwn+UYpetA4lo7GLKye0j6MwRA+33p3RR0Mwit/NnuhO3s46wACt07PhVppwU/pw0H9zIQWQBtVWQNpDHZ5DW/InAssSRoQ9cPQOdAaxWk6xUVDBfkQOI9yhxG98HOwfxgcReRMfhBHAM+Q7JHUJeJ6zVbXU4ysJyTDwGrMHAYZrckkLncMpq5xNeR6iIX4TdYCHRqRXMEzhB+EtuzWYRAmvezm/JYwad22PyzbMUEkoOchzL97rpXHrZPsJOblFB0Cq5TOFXg+YoMYxPPejzEkVGwMEu2Tqu9wJ1v+YVnklx9e24/C6fpddGzlGPAVwZiQZ9xwydn7pb9Yi/QaNk2SFCvDDmD4ACkjX7o4P9tr9YLs2xB+fKfVPfVjwQiWrXBTY9e33UQXMGbWGx5BBY7tPalesOIk2QHm0kVhWTyyyW1MRDsMKkDyA4141b/cig943hKbgYDvQ+I0LDqOFDrYktNGvgY9DgI/texZsTFPorrjcseEONkwhWqpix4ZZBT0ypu+Pu2DFIgBkfW1jlJ5L6HAKUUJ6+eGo0w+0oEP79OyOw2NgO+2sFuPDPRchCCppXsFsYvc2AGYQBKqWh6/mXVaVtnD1DHSlS3lnIeGmgZ28mpteH8Olxy2e59UUu8ASzkpevlYcwvPl62YPEMZQFmknjEzN9Qxt3f1DZdb6xcOVbRWghF1Tye+ZzOiNRQH4hFJiLT6BIL6Afq+txzfd1LLM+gdY5WNWg1iZLVwqXEN0sYHoDZb/9Q9COZxuT+qaBcIfGEOEBAWrg5zDnGU3dXqCxgBlHTPFpGSr4Nrsuh4ZDha7zBdjBsPgpmXAa4ixrGEhWTnnWJHs8a+6SGGyGBUMPJFHiNjUCgNee0ZVxFAHoTisQbGwAAitPrSdzkFrnGHGzZMp1/ccIqDnhOa4Lb7vHU2G/qvSWSPPd6lefZdqi0Z9gM1rwJC9eH8t/Z3W//3PPsG+PQO1IcfvhIF2urN+5HyO4TaDmsChBPzEoES6aQSeRv3NBlf34rcRMf/Qz5UBj5xj0fWJhwmGQ1wX6QyquRfLvSlezQkPToOJdUFHZFfS6IV1ZrtgCx89aaDs/PyslQPSAGoUjtye2f8Lb3N3DZUgeCZ4YmInHWXVgEyrl9pMJzCTWLB7GXUunpTDwXbDYOdMiTshVbr3HcE6p7BzbJIJCTW6Aoxfo7Lr2/fGxTd4Dtwdj8EJFFgonVTfOrDRFgng3hmOjZSvzb2C71ydSV77CexrW1kQfJqNdhSTyzIp0N8h+cuY0May9JlRc3PSdJ7BcV/yX5egnSKe87jluVOKJoyd9GDde2VFWtvaSEXRGek6iY0zHLzXj5gxkUMsKfrcEE23lWW88pPiARsfUXpJW10wi9YP8z20mXmNGrjLdZN/QgAc5CGfNKQWOfoIA30GQJQTvZCcQ7ApjnMrqDYwnHujccsm2oxUiByzqzP52KJt+WzOQ9/jhr2AbVy7LOty7lmudyg+gF1flEyZThd0N/OG3tQ3x9ndJJg7S+gQ2hh5QY6rpVR8iGbF2+umD3NZ0kEz2n2+WoC3rQzod488mb2swVtuybNlCh96t923TazfwmmVrqCcxF2+Hw07tRTG7KZid6VKO9JC8cgQPCPTbjNo2wB8hipleJmXM1HEO4Sf4QWcqwqdIML8GJHH9sv1D1vT7oebpx7nUxvLI3IFLYMCiQsQA5A+Is09LM60Dzwb+svXyESou28dYnxmRKULsBjE6XyX3PQL5MLttni8NfxudwbXrelOF/3fJ6Iua6R3AdJ1XlOsXIrjtp5DVQptKlaQXKRQ7VZedttONtF7Mjk9Sz59vJokF5PLL5+uLpPLL6enRxf/igvofmKXj1i+6b9LGn+toNWz4mBPgt+23DiZIwpWihlXurkSN+o0qD3qQnZHXyM3rva02Qw1M4d0Cbcj8agTSNc79nQaJWqTG+kU6MgzewNbQpNjQ94NFBvVEO+msB6Gna6MzgVe9Hd/avAT4JPtZdZ/pQi2gYc3pK2HUWPY3jWk/+gajgKiIzAdgii1RW3zOxRe+fnfpOIjOV9iS3BuVgLoXlMpajPRJ0lWpUkSdihjlmUJcyQBHQ6trHgPYAtn1qn6O0g6bcgQI3A7sacwM4tlZP5DVirwapvyPf5ua9VjGFuZNzurcbtlfanTANYSrBmYy/VsWdQqsCdFYGQAPD0+CHs+2QNnQJlNEgyfJCFjaFOSBF2TJK5LsWX0cqU0LyYPQgfWceHgf1BLAwQUAAAACAApr/tcjAPrT7kKAAAMIwAAIQAcAEJhY2tFbmQvc2NyaXB0cy9ydW5fY29tcGFyaXNvbi5weVVUCQADvmNnaqVmZ2p1eAsAAQT1AQAABBQAAADNGmtv3Ljx+/4KVgcUkrvW2YcG7QndA3KJgwZtEsNJCxQLg+BK3DXPeh0p2d66+987w4dEPXbt66cKQaLlPDhvzlAJguDqgeUtazhp7jjZtnlOrq+//OGbZKXaVrLgkhS8uasywnZMlKoBPMkBUzzxjGyY4rkouYqDIFgstrIqCKXbtmklp5SIoq5kQ1hZVg1rRFWqxcKtyV3NpOLu9y+qKt17wZo79672yrBNqzznqWYSs03qeL9jec42OTdINVDmYuOA18hIA5p9LcqdW39b7heLn9+++9vV5/f05suXb2SlcUOQXeQgeRRLrqr8gYdRDGLyslHry9uF2BLVyNCnjAjoRkSJgsa4fbIg8LhfMZiMyya8WE4pI2swJdOYP9VcigI3irnxCGjq5A0J+Q72+ZUl5OqPFz/oHZzpKdNGobWsHkTG5VJDLQ9OeS1UlXGzWrB7WCkfhKxK3MuuAjyfZ/MoBfDIeMPALBlN1cNyEc1LLarO3U1ViJQaWvTrEoRN73mZUTTJkuxEQyGoWMYatiR5xTK6Z0U+UnJ2F5XykklRUYgRKZ7clptW5BkdAZcYSEDJcvFv3gFftQtGq2pEqk554K7KC8qyX1plDVkzIcFKjyJPqydQ20RCWxSwL0iALuEKDbj4dPXtr1/ef4WwCwMmgiUJNhLMkDLV4I8MNmdlyvG9lTtepvsgWnz9B2Tmzber9/T9l09vP3425Hdid/fI9gZ1w0pAvL75+Ontzb8o7HLz8Z1G06IEBWclzcEowJAWKjASBny7xcx6QE/n8I/cUwlIDsw0GBR74HcihfRIK8Bhuw4hrYqiLUWqY5Yi8I6zzEEbMGq+p/wBTOv4Rot3EL43bVlCfVl1SbxeX+sAgVRZQk419g0XyX/I56rkS5KJtFnrZcji29vJwmKxyPiWyLakYE5uNO8iQ2eoZmiky6oCqlqi9zGYnGeJ3tvkhq59HtwkS8/GyaWBaVVuxS4ZiQTqkvOfRoumTHi5CGYYp2c4EHtphV1qGZd2s0jzaeTeMMQHqpQt2asVwfDqQQ7cKUGE0vKTSupC1kNioXQxDKMhOT6SCcXJB4B+rpoPVVtmV1JWMtwGDYBKOBg0H2ReCKWg8ibkuWd9CKIBy1HpQUvMlaSw57D0Ldcz47niyQusj9XN0NisZyY5nGLlpJKGA/6TYnpk4yEQDhfeUHTjSvtyAFRsy5s9VJEyqx4hS1dbqJBNaNwd73gTBmMUSP7Li4so6hkZNbaihMzyQsMTN07zCtIjsvkCvi6xXpsiFcrqUSUkhzq0niQcGEqKVOeEDmyNpYW0UW0tt+62NRoAz7Whve2tDD0GAQCeoLinH8TwW6traCKMJgxRHa6szHSbAFFqBA+ne5hNXEHo6zDuc0LBo8m6k1VbY3XogSNSCK/nw8KpZZMQNLPlvveDAWlJgGaNBhgZwhpgHRjU4BaT2bzfdmysRGu7jtsPQqnz1OgMCs0/wyy0IgMByhBOkv7I6TFAqd9cvITx45sXMX58AQNPlKpt/EPKf85G59/6MrkdYkXjgmgNQpLVOBE8T7nIj3ryw8IEPHuktaXww8MkRR8VGfQHpoX1sTZVlY9CxxUpL3hQi2SQNr2zRgqPSqCgm70+CyfxoWXHIEMoFRBlyalc9Tefy1nPoJPQxXNIZ+2JrB4wOgx+dVX7/0CVzjn/uzq2UxQZpr+CBpNnYe+n+J7vVRiR30/UtpChHrAGXLZgYfqg6LOjOSTPRqhDMA73fvvp2Z7zLfYi616cNaRDcatNhW9oq57B7dQL0JBqDmPhfysfL6nWoCOmyKjDDlHYpdlx6tu8euSSQhuz4U1j2oouY56n1exoH3uYcO7z2Ak2LZf4hFB4tYwR+QsW4dBIGqEPxtJh7+IT/OQTTLibFTN8QLu18oeR0LecQVRiV4otaAZDxSB1QPxklEmBJQ2SqQuGZTTwNgWv1EChcUdYGElaxyAZW26E2YvZOF5guIv44s3SK7nuDQMJmCwtInjViRJjfEGeLHoC25M820xWwN+enuD4UVhR1AsQfKsd/LmiKmo4TmGyDU+2/QiDc6puG90swajQDx5nSysWeJkn+gwAx3xgEAV2lsA6JvWElJDBtORGmxNzhdmVGmtXcq/73n4EDw08msWNi3t4D+21x+qbBIcS/gSdEq3u9U9Dd7yDwuQ3+bxlIm8ln+LBXz6emWywGDZtDTOH3+xaGCToZAC2iQHZhDXXMYGwYOU+ND/d9cx0eMbwsTjwxxJ7w44Zcv6JsWWmGycJKSDI0OkN0tr5Gw+D74Gjm8GtXDkvQ+j1Q8c+Ir9b6cXfvB/q4fbMwEhYpriyO83q0rPemBOzu2mx9l0HCOivTcyqCm7XhoN3XuJVDegBTBypXQn6wg3pga0TJjfG29yVzLBMunmWfIf3c5AFkG+V5Gsmd+e4MDwTNkyNKoseNkHqtmxWMLGHViQTNg5oVFqSN4P5CB8oGyIztf4IjwGGY/SnMR9wQ3NUCgNztJczQ5pjoSeB3obGp/1P9KtnYIit/mesIB4a3WYhJ88n3qQ/KgHOjxpj1udTAbSYw5YBkXCGRXDB6lDf2jjeWnuEAv+ZawTbqunm5dlsD02Lr5aLHpEdEuRDn/HvUUPTsbJqzp/F4zI3i/Q9Mf1jesfT+7oCZdR0L4No5D0GnNXhCPI28FSL8cJ0uue0A8DOVB8eug3t9D91bYMPVu2YP0Hbk4W4VYxFQYU9vYSWhzaAEUJnVGWi3K2Cttme/zmIonWA5MFMH40PVidRtnwCHNxO+c/gfktfgE3va+e9iU9v5OkM6J5XeR0f8Lzb8YjLDdYJt1uE17veEAzcPzPOuufIdZF7jvhEdxH2mmHaY+PzwlXF+DE9NPLzOpTjTtKSD64xT6LaI+E0u6M28GQEdV7A8e5Axxemphnv73WPPabOHceZ94mmdH6JWV1jLp7cZzrqjp9u7E1cVT0tuaYxxsau/BVW1xReRCPZbKi/hg14EPvrFx2pse3snbzKrficndkgPY06nevcM++3yXetvmouYbCYeoCYcpn03j7Mc/aLcoc7ReVPKa8bcqX/wSOZKVxL+m9TP//96uLikpyTGq+Y5YP5qis5uhe/f5oubL4M2FbdBeS8Qhw7U1iBAxs7tBC2j2JKS1ZwSg8wSsLCIbBqTr8ehpPBBOpuh4DnWd6oGBChV+ptgJ/m9AcolEMNTni/lSZn+N0mnDSoRzsxIED62W5l0ceBuT3dD+bmAIcwLCFoHW9Y6ZbBVvj5nOWB1/AFA01MzwgMhvrNoRvLUBtOI3sYLWzh9nfDOTXnU3pE19b1cJ370bf2tYeenc3cnvvUO4F6+N91Qws2STZNnblAsIY2LdDS2T3y53e75j5a6P9CQGFeaPWH21APxO6/FsSfISpVzVL7TUgv4gDdIby1hNcaEmZcpVLo1FpRmlUppZFHGbMs6/YKg/Nz43gMVf5rizcI3nR8hMSo/ZtITJcHJOaT0ipQYC9OG6ALBrax9J1NVPdxB/PD2AYSxBijsxnYY2rGUdyPLj36ObIjie3XyGXPODa6Lm2buuoBZsEqLTFndSeatUWtQrsrfv/NAHv1QzRQ8hJPa4uz7qP21hzZF6AwwF1B0uc5pag+pfYzqBmzv+5Vw4urJ9GExjjR4r9QSwECHgMUAAAACABFq/tc7JaunykJAAAMIAAAKQAYAAAAAAABAAAApIEAAAAAQmFja0VuZC9zY3JpcHRzL3J1bl9kZW1vX2xpdGVfcGlwZWxpbmUucHlVVAUAA3JcZ2p1eAsAAQT1AQAABBQAAABQSwECHgMUAAAACACRq/tcCmP3ongKAAC8HAAALQAYAAAAAAABAAAApIGMCQAAQmFja0VuZC9zY3JpcHRzL2dlbmVyYXRlX2RlbW9fbGl0ZV9yZXN1bHRzLnB5VVQFAAMBXWdqdXgLAAEE9QEAAAQUAAAAUEsBAh4DFAAAAAgAKa/7XIwD60+5CgAADCMAACEAGAAAAAAAAQAAAKSBaxQAAEJhY2tFbmQvc2NyaXB0cy9ydW5fY29tcGFyaXNvbi5weVVUBQADvmNnanV4CwABBPUBAAAEFAAAAFBLBQYAAAAAAwADAEkBAAB/HwAAAAA="
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BOOTSTRAP_B64))) as archive:
    archive.extractall(REPOSITORY)
assert PIPELINE.is_file()
print("Installed embedded demo-lite pipeline scripts.")

# Install project dependencies first, then a current self-contained SUMO wheel.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "./BackEnd"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "eclipse-sumo==1.27.1"], check=True
)

sumo = importlib.import_module("sumo")
SUMO_HOME = Path(sumo.SUMO_HOME).resolve()
SUMO_BIN = SUMO_HOME / "bin"
SUMO_TOOLS = SUMO_HOME / "tools"
os.environ["SUMO_HOME"] = str(SUMO_HOME)
os.environ["PATH"] = str(SUMO_BIN) + os.pathsep + os.environ.get("PATH", "")
os.environ["PYTHONPATH"] = str(SUMO_TOOLS) + os.pathsep + os.environ.get("PYTHONPATH", "")
print("SUMO_HOME:", SUMO_HOME)
print("sumo binary:", shutil.which("sumo"))
print("netconvert binary:", shutil.which("netconvert"))
version = subprocess.run(["sumo", "--version"], capture_output=True, text=True, check=True)
print(version.stdout.splitlines()[0])

# Force the fixed three-lane highway to be written directly. This removes all
# dependence on netconvert, including broken /usr/bin builds cached by Colab.
source = GENERATOR.read_text(encoding="utf-8")
start = source.index("def _write_network(")
end = source.index("\ndef _write_routes(", start)
network_writer = """def _write_network(config: ScenarioConfig, artifacts: ScenarioArtifacts) -> None:
    speed_mps = config.speed_limit_kmh / 3.6
    lane_width_m = 3.2
    root = ET.Element("net", version="1.9", junctionCornerDetail="5", limitTurnSpeed="5.50")
    ET.SubElement(
        root, "location", netOffset="0.00,0.00",
        convBoundary=f"0.00,0.00,{config.road_length_m:.2f},0.00",
        origBoundary=f"0.00,0.00,{config.road_length_m:.2f},0.00", projParameter="!",
    )
    edge = ET.SubElement(root, "edge", id="highway", **{"from": "start", "to": "end"}, priority="1")
    lane_ids = []
    for lane_index in range(config.lane_count):
        lane_id = f"highway_{lane_index}"
        lane_ids.append(lane_id)
        lateral = -(config.lane_count - lane_index - 0.5) * lane_width_m
        ET.SubElement(
            edge, "lane", id=lane_id, index=str(lane_index), speed=f"{speed_mps:.6f}",
            length=f"{config.road_length_m:.2f}",
            shape=f"0.00,{lateral:.2f} {config.road_length_m:.2f},{lateral:.2f}",
        )
    road_width = config.lane_count * lane_width_m
    ET.SubElement(
        root, "junction", id="end", type="dead_end", x=f"{config.road_length_m:.2f}", y="0.00",
        incLanes=" ".join(lane_ids), intLanes="",
        shape=f"{config.road_length_m:.2f},{-road_width:.2f} {config.road_length_m:.2f},0.00",
    )
    ET.SubElement(
        root, "junction", id="start", type="dead_end", x="0.00", y="0.00",
        incLanes="", intLanes="", shape=f"0.00,0.00 0.00,{-road_width:.2f}",
    )
    ET.ElementTree(root).write(artifacts.network, encoding="utf-8", xml_declaration=True)


"""
source = source[:start] + network_writer + source[end + 1 :]
GENERATOR.write_text(source, encoding="utf-8")
section = source[source.index("def _write_network(") : source.index("def _write_routes(")]
assert "netconvert" not in section
subprocess.run([sys.executable, "-m", "py_compile", str(GENERATOR)], check=True)
print("Applied direct-network compatibility fix to:", GENERATOR)

# Write a compact but presentation-useful protocol.
training_config = """run_mode: demo_lite
domain: highway
base_training_config: configs/training_config.yaml
base_scenario_config: configs/scenarios/highway_emergency.yaml
safety_window_ms: 100
dataset:
  train_configs: 3
  validation_configs: 2
  test_configs: 4
training_seeds: [101]
training:
  episodes_per_config: 15
  validation_interval_episodes: 5
  early_stopping_patience_episodes: 10
  evaluation_episodes: 1
champion_selection:
  enabled: true
  shortlist_size: 3
  evaluation_episodes: 1
reward_profile: balanced
reward_profiles:
  balanced:
    effective_delivery: 0.25
    coverage: 0.30
    latency: 0.20
    overhead: 0.10
    missed: 0.10
    fairness: 0.05
environment_overrides:
  max_vehicles: 50
  max_events: 2
  feature_mode: basic
ppo_overrides:
  n_steps: 16
  batch_size: 16
  n_epochs: 3
  verbose: 0
  transformer:
    variant: standard
    num_layers: 1
    num_heads: 2
    d_model: 64
    feedforward_dim: 128
    dropout: 0.1
success:
  reward_threshold: 0.5
  convergence_rate: 0.34
"""
comparison_config = """run_mode: demo_lite
domains: [highway]
safety_window_ms: 100
base_scenario_configs:
  highway: configs/scenarios/highway_emergency.yaml
models:
  highway: experiments/demo_lite/highway_batch/champion/model_best.zip
dataset:
  train_configs: 3
  validation_configs: 2
  test_configs: 4
test_seeds: [1701, 1801]
environment:
  episode_steps: 10
  max_vehicles: 50
  max_events: 2
  critical_radius_m: 300
  road_length_m: 5000
  lateral_extent_m: 10
  delay_normalization_ms: 100
  feature_mode: basic
network:
  highway_mode: simple
  carrier_frequency_ghz: 5.9
  transmit_power_dbm: 23
  noise_floor_dbm: -94
  sinr_midpoint_db: 5
  sinr_scale_db: 2
  max_queue_delay_ms: 50
reward_weights:
  effective_delivery: 0.25
  coverage: 0.30
  latency: 0.20
  overhead: 0.10
  missed: 0.10
  fairness: 0.05
"""
(REPOSITORY / "BackEnd/configs/batch_training_demo_lite.yaml").write_text(
    training_config, encoding="utf-8"
)
(REPOSITORY / "BackEnd/configs/comparison_demo_lite.yaml").write_text(
    comparison_config, encoding="utf-8"
)

# Full scenario preflight: network, routes, TraCI simulation, trajectory and events.
PREFLIGHT = Path("/content/g11-demo-lite-v2-preflight")
preflight = subprocess.run(
    [sys.executable, "-B", str(GENERATOR), "--output", str(PREFLIGHT)],
    cwd=str(REPOSITORY),
    capture_output=True,
    text=True,
)
print(preflight.stdout)
if preflight.returncode != 0:
    print(preflight.stderr)
    raise RuntimeError("SUMO scenario preflight failed; training was not started")
for required in ("highway.net.xml", "trajectory.xml", "events.json"):
    assert (PREFLIGHT / required).is_file(), f"Preflight output missing: {required}"
print("Scenario preflight passed.")


def pipeline_command(*extra):
    return [
        sys.executable,
        "-B",
        str(PIPELINE),
        *extra,
        "--persistent-root",
        str(DEMO_LITE_ROOT),
        "--work-root",
        str(DEMO_LITE_WORK),
    ]


def read_status():
    result = subprocess.run(
        pipeline_command("--status"),
        cwd=str(REPOSITORY),
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("Unable to read demo-lite status")
    return json.loads(result.stdout)


# One notebook execution advances training -> comparison -> result generation.
# Code 75 is a safe checkpoint pause; this loop immediately resumes it.
for attempt in range(20):
    status = read_status()
    print(json.dumps(status, indent=2, ensure_ascii=False))
    if status["next_stage"] is None:
        break
    command = pipeline_command("--stage", "next", "--time-budget-minutes", "240")
    print("Starting/resuming stage:", status["next_stage"])
    process = subprocess.Popen(
        command,
        cwd=str(REPOSITORY),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code == 75:
        print("Safe checkpoint reached; resuming automatically...")
        continue
    if return_code != 0:
        print("Saved failure diagnostics:")
        for failure_path in sorted(DEMO_LITE_ROOT.rglob("status.json")):
            try:
                value = json.loads(failure_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if value.get("status") == "failed":
                print(failure_path)
                print(json.dumps(value, indent=2, ensure_ascii=False))
        raise RuntimeError(f"Demo-lite stage failed with return code {return_code}")
else:
    raise RuntimeError("Automatic resume limit reached")

final_status = read_status()
assert final_status["next_stage"] is None, final_status
print(json.dumps(final_status, indent=2, ensure_ascii=False))
print("DEMO-LITE PIPELINE COMPLETED")

In [ ]:
import pandas as pd
from IPython.display import Image, display

RESULTS = DEMO_LITE_ROOT / "presentation_results"
TABLE = RESULTS / "table_comparison.csv"
METRICS_FIGURE = RESULTS / "fig_metric_comparison.png"
TRAINING_FIGURE = RESULTS / "fig_training_curve.png"

assert TABLE.is_file() and METRICS_FIGURE.is_file() and TRAINING_FIGURE.is_file()
display(pd.read_csv(TABLE))
display(Image(filename=str(METRICS_FIGURE)))
display(Image(filename=str(TRAINING_FIGURE)))

archive = shutil.make_archive("/content/G11project-demo-lite-v2-results", "zip", DEMO_LITE_ROOT)
print("Persistent results:", RESULTS)
print("Downloadable archive:", archive)